In [1]:
import tkinter as tk
import random
import math
from copy import deepcopy


# =========================
# GAME LOGIC
# =========================
class ConnectFour:
    def __init__(self):
        self.board = [[0] * 7 for _ in range(6)]
        self.current_player = 1
        self.game_over = False
        self.winner = None

    def get_valid_columns(self):
        return [col for col in range(7) if self.board[0][col] == 0]

    def make_move(self, column):
        for row in range(5, -1, -1):
            if self.board[row][column] == 0:
                self.board[row][column] = self.current_player
                break

        if self.check_win(self.current_player):
            self.game_over = True
            self.winner = self.current_player
        elif not self.get_valid_columns():
            self.game_over = True
        else:
            self.current_player = 3 - self.current_player

    def check_win(self, player):
        # Horizontal
        for row in range(6):
            for col in range(4):
                if all(self.board[row][col + i] == player for i in range(4)):
                    return True

        # Vertical
        for row in range(3):
            for col in range(7):
                if all(self.board[row + i][col] == player for i in range(4)):
                    return True

        # Diagonal /
        for row in range(3):
            for col in range(4):
                if all(self.board[row + i][col + i] == player for i in range(4)):
                    return True

        # Diagonal \
        for row in range(3, 6):
            for col in range(4):
                if all(self.board[row - i][col + i] == player for i in range(4)):
                    return True

        return False


# =========================
# MCTS NODE
# =========================
class MCTSNode:
    def __init__(self, game_state, parent=None, move=None):
        self.game_state = game_state
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0
        self.visits = 0
        self.untried_moves = game_state.get_valid_columns()

    def expand(self):
        move = self.untried_moves.pop(random.randrange(len(self.untried_moves)))
        new_game = deepcopy(self.game_state)
        new_game.make_move(move)
        child = MCTSNode(new_game, self, move)
        self.children.append(child)
        return child

    def select_child(self, exploration=1.414):
        def ucb(child):
            if child.visits == 0:
                return float('inf')
            return (child.wins / child.visits) + exploration * math.sqrt(
                math.log(self.visits) / child.visits
            )

        return max(self.children, key=ucb)

    @staticmethod
    def run_mcts(game_state, iterations=1000):
        root = MCTSNode(deepcopy(game_state))

        for _ in range(iterations):
            node = root

            # Selection
            while node.untried_moves == [] and node.children and not node.game_state.game_over:
                node = node.select_child()

            # Expansion
            if node.untried_moves and not node.game_state.game_over:
                node = node.expand()

            # Simulation
            sim = deepcopy(node.game_state)
            while not sim.game_over:
                moves = sim.get_valid_columns()
                if not moves:
                    break
                sim.make_move(random.choice(moves))

            # Result
            if sim.winner is None:
                result = 0.5
            elif sim.winner == game_state.current_player:
                result = 1
            else:
                result = 0

            # Backpropagation
            while node is not None:
                node.visits += 1
                node.wins += result
                node = node.parent

        if root.children:
            return max(root.children, key=lambda c: c.visits).move

        return random.choice(game_state.get_valid_columns())


# =========================
# GUI
# =========================
class ConnectFourGUI:
    def __init__(self, root):
        self.root = root
        self.game = ConnectFour()

        self.root.title("Connect Four")

        self.canvas = tk.Canvas(root, width=490, height=420, bg="blue")
        self.canvas.pack()

        self.status = tk.Label(root, text="Your Turn (Red)", fg="red", font=("Arial", 14))
        self.status.pack()

        # Buttons
        button_frame = tk.Frame(root)
        button_frame.pack()

        for col in range(7):
            tk.Button(button_frame, text=str(col + 1),
                      command=lambda c=col: self.human_move(c),
                      width=4).pack(side=tk.LEFT)

        # Difficulty
        self.iterations = tk.IntVar(value=1000)
        tk.OptionMenu(root, self.iterations, 100, 500, 1000, 2000).pack()

        tk.Button(root, text="New Game", command=self.reset).pack()

        self.draw()

    def draw(self):
        self.canvas.delete("all")
        for r in range(6):
            for c in range(7):
                val = self.game.board[r][c]
                color = "white" if val == 0 else "red" if val == 1 else "yellow"
                self.canvas.create_oval(c * 70 + 5, r * 70 + 5,
                                        c * 70 + 65, r * 70 + 65,
                                        fill=color)

    def update_status(self):
        if self.game.game_over:
            if self.game.winner == 1:
                self.status.config(text="You Win!", fg="green")
            elif self.game.winner == 2:
                self.status.config(text="AI Wins!", fg="red")
            else:
                self.status.config(text="Draw!", fg="gray")
        else:
            self.status.config(text="Your Turn (Red)", fg="red")

    def human_move(self, col):
        if self.game.game_over or self.game.current_player != 1:
            return

        if col not in self.game.get_valid_columns():
            return

        self.game.make_move(col)
        self.draw()
        self.update_status()

        if not self.game.game_over:
            self.status.config(text="AI thinking...", fg="orange")
            self.root.after(100, self.ai_move)

    def ai_move(self):
        move = MCTSNode.run_mcts(self.game, self.iterations.get())
        self.game.make_move(move)
        self.draw()
        self.update_status()

    def reset(self):
        self.game = ConnectFour()
        self.draw()
        self.update_status()


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    root = tk.Tk()
    app = ConnectFourGUI(root)
    root.mainloop()